In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Users/utkarshadlakha1702@gmail.com/DataBricks_DataWarehousing/consolidated_pipeline/1_setup/Utilities


In [0]:
print(bronze_schema)

In [0]:
#creating widgets for better visiblity
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","customers","Data Source")

catalog=dbutils.widgets.get("catalog")
data_source=dbutils.widgets.get("data_source")
print(catalog,data_source)
#displaying the

In [0]:
# read data path from s3
base_path = f's3://child-company-dbproj/{data_source}/*.csv'
df = (spark.read.format("csv")
      .option("header", True)
      .option("inferSchema", True)
      .load(base_path)
      .withColumn("read_time", F.current_timestamp())
      .select("*","_metadata.file_name","_metadata.file_size")
      )
display(df.limit(10))

In [0]:
# Write all data as is to bronze layer
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

display(df.limit(10))

### SILVER LAYER PROCESSING



In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")
df_bronze.show(10)



In [0]:
# Checking for duplicates
df_duplicates = df_bronze.groupBy("customer_id").count().filter(F.col("count")>1)
display(df_duplicates)

In [0]:
# Removing duplicates
print("Rows before", df_bronze.count())
df_silver = df_bronze.dropDuplicates(["customer_id"])
print("Rows after", df_silver.count())

In [0]:
#Check for leading and trailing spaces
display(
    df_silver.filter(F.col("customer_name")!=F.trim(F.col("customer_name")
                                            )))


# Replace those params with trimmed params
df_silver = df_silver.withColumn("customer_name",F.trim(F.col("customer_name")))


In [0]:

#Check for incorrect mislead data in city
df_silver.select('city').distinct().show()


#mapping accroding to the correct names

city_mapping = {
    'Bengalore':'Bengaluru',
    'Hyderabadd':'Hyderabad',
    'Hyderbad':'Hyderabad',
    'NewDelhee':'New Delhi',
    'NewDelhi':'New Delhi',
    'Bengaluruu':'Bengaluru',
    'NewDheli':'New Delhi'
}

allowed_cities = ["Bengaluru","Hyderabad","New Delhi"]

#inserting this data to silver

df_silver = (
    df_silver.replace(city_mapping,subset=['city'])
    .withColumn(
        "city",
        F.when(F.col("city").isNull(),None)
        .when(F.col("city").isin(allowed_cities),F.col("city"))
        .otherwise(None)
    )
)
# show after correcting
df_silver.select('city').distinct().show()

In [0]:
# Checking conditions for customer name
df_silver.select('customer_name').distinct().show()

df_silver = df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(),None)
    .otherwise(F.initcap("customer_name"))
)

#show after convering to correct casing
df_silver.select('customer_name').distinct().show()

In [0]:
# checking for null cities
df_silver.filter(F.col('city').isNull()).show(truncate=False)

In [0]:
null_city_customer_names=[row['customer_name'] for row in df_silver.filter(F.col('city').isNull()).select('customer_name').collect()]
null_city_customer_names



In [0]:

# check for city of same customer in other records
df_silver.filter(F.col("customer_name").isin(null_city_customer_names)).show(truncate=False)

customer_city_fix = {
    789403:"New Delhi",
    789420:"Bengaluru",
    789521:"Hyderabad",
    789603:"Hyderabad"
}

df_fix = spark.createDataFrame(
    [(k,v) for k,v in customer_city_fix.items()],
    ["customer_id","fixed_city"]
)
df_fix.show()

In [0]:

df_silver = (
    df_silver.join(df_fix,"customer_id","left")
    .withColumn("city",F.coalesce("city","fixed_city"))
).drop("fixed_city")
df_silver.show()

In [0]:
#check for fixed issues
df_silver.filter(F.col("customer_name").isin(null_city_customer_names)).show(truncate=False)

In [0]:
#convert customer id to string for gold layer
df_silver = df_silver.withColumn("customer_id",F.col("customer_id").cast("string"))
print(df_silver.printSchema())

In [0]:
# in our gold layer of parent company we dont have a city column so we need to combine name and city into one column and also add platform name and channel as to match the schema of gold.dim_customers
df_silver = (
    df_silver
    # Build final customer column: "CustomerName-City" or "CustomerName-Unknown"
    .withColumn(
        "customer",
        F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )
    
    # Static attributes aligned with parent data model
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)





In [0]:
display(df_silver)

In [0]:
#write to schema now

df_silver.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

In [0]:
#filtering out columns that are required (Gold processing)

df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source}")

df_gold = df_silver.select("customer_id","customer_name", "city", "customer", "market", "platform","channel")
display(df_gold)


In [0]:
#write it to gold schema
df_gold.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")


In [0]:

delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)


In [0]:
# merging part upsert
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()